# Building a Chain

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_cohere import ChatCohere 
from langchain_huggingface import ChatHuggingFace

c:\Users\delta\RAG\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\delta\AppData\Local\Temp\ipykernel_14928\1173050485.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
video_id = "J5_-l7WIO_w"

try:
    yt_api = YouTubeTranscriptApi()
    transcript_manager = yt_api.list(video_id)

    transcript_data = transcript_manager.find_transcript(['hi'])
    transcript_list = transcript_data.fetch()

    transcript = " ".join([chunk.text for chunk in transcript_list])

    print("Transcripts Fetched Successfully!")
    print(transcript[:200])

except TranscriptsDisabled:
    print("Transcripts are disabled for this video.")
except NoTranscriptFound:
    print("No transcript found for this video.")
except Exception as e:
    print(f"An error occurred: {e}")


Transcripts Fetched Successfully!
हाय गाइज़, माय नेम इज नितेश एंड यू आर वेलकम टू माय YouTube चैनल। इस वीडियो में भी हम लोग अपना लैंग चेन प्लेलिस्ट कंटिन्यू करेंगे। अह पिछले वीडियो में हमने रैग पढ़ना शुरू किया था और हमने फोकस किया था रैग


In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [6]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7619.42it/s]


In [7]:
vectorstore.index_to_docstore_id

{0: '4baa25f2-4327-41f9-9b41-4237c77f9136',
 1: '08c96ba8-ecf4-489d-9355-f76835513c1e',
 2: 'c0ed317c-91bf-4cb4-a37d-32670d5fbbd9',
 3: '8bc1c3f9-1a8b-4d80-ba31-df658b0744da',
 4: '31aa28ed-93be-4aa7-aed8-5dceee2b5d6c',
 5: 'a70f668d-fc2b-4c63-b5a1-c66d5702c2c5',
 6: 'b81caa48-be2b-4a9b-843d-638c9c6a501a',
 7: '7dd15e64-2c02-4d23-91b5-4f5758bf9f67',
 8: 'ed9caabe-3be6-4d84-80a0-0ba675d46a7d',
 9: '562f70f7-e396-48ee-bf32-85ab0afdb628',
 10: 'caa4319a-5e41-4ed7-9be2-7c4757e3f838',
 11: '88d23ad0-2eff-4745-a4a2-30c4553656d2',
 12: 'ee6cc4fa-ad97-4e14-874c-c3c2020a7fa5',
 13: 'd0e593f5-6556-40b2-8bdd-182f40eef267',
 14: '7fc33cf8-60e5-4478-bb7c-83e8449877f8',
 15: '156e28a4-5f00-488f-a0a2-cddbc9403c77',
 16: '02cebb39-9bdd-40b8-be9e-f02ff83e7d5e',
 17: '1f29d515-eabb-4810-8f97-c462a958848a',
 18: '928a22c9-79e1-4277-9ee8-8f2e62e02699',
 19: '52408b9a-628b-45a7-8914-97c533153bca',
 20: '60202d37-af6a-4429-bc30-c697f768c8b3',
 21: '4e9d87f5-fce1-45c4-9539-c91eabec5d92',
 22: 'eddc6982-f61e-

In [8]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":4})

In [9]:
retriever.invoke("What is RAG?")

[Document(id='02cebb39-9bdd-40b8-be9e-f02ff83e7d5e', metadata={}, page_content='है, यह 2 घंटे का वीडियो का ट्रांसक्रिप्ट है। तो, ऑब्वियसली बहुत लंबा होगा। हमें इसको छोटे-छोटे चंक्स में डिवाइड करना है। और यह काम करने के लिए हम लोग एक टेक्स्ट स्प्लिटर को यूज करेंगे। और हम जो टेक्स्ट स्प्लिटर यूज़ कर रहे हैं उसका नाम है रिकर्सिव कैरेक्टर टेक्स्ट स्प्लिटर। ठीक है? चंक साइज फिलहाल मैंने 1000 रखा है। चंक ओवरलैप मैंने 200 रखा है। और इस कोड को मैंने रन किया और फिलहाल मेरे पास 168 चंक्स बन गए हैं। ठीक है? तो यहां पे आप अलग-अलग चंक साइज़ज़ एक्सपेरिमेंट कर सकते हो। फिलहाल मुझे इस पर्टिकुलर वैल्यू से सही रिजल्ट्स मिल रहे थे। तो आई एम कीपिंग इट एज ₹1200। ठीक है? अगर आप कोई एक पर्टिकुलर चंक देखना चाहते हो तो आप चंक्स में उसका आईडी डाल दो और आपको वो चंक दिखने लग जाएगा। ठीक है? तो यह जैसे 100 नंबर वाला चंक है। तो इस पॉइंट पे हमने स्टेप नंबर टू भी कर लिया है कि हमारे पास हमारे पूरे के पूरे ट्रांसक्रिप्ट के छोटे-छोटे चंक्स आ गए हैं। ठीक है? नेक्स्ट स्टेप में हमें क्या करना है कि इन सारे चंक्स को हमें वेक्ट

In [10]:
llm = ChatCohere(model="command-xlarge-nightly", temperature=0.7, max_tokens=3000)

In [11]:
prompt = PromptTemplate(
    template = """ You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}""",
      input_variables = ["context", "question"]
)

In [12]:
question = "What is RAG?"
retrieved_docs = retriever.invoke(question)

In [20]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [21]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text


In [23]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [25]:
#parallel_chain.invoke({"question": question})